# Web App Back End Tests

Tests for the back end of the Structure Relations web application.
Uses *RS.GJS_Struct_Tests.BRBL BH.dcm* file located in the *test_data* folder.

## 1. Import Required Libraries

First, let's import the necessary libraries including our custom DicomStructureFile class.

In [1]:
# Import required libraries
from typing import List, Tuple
#import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import shapely

# Add the src directory to the Python path
#sys.path.append('src')

# Import our custom DicomStructureFile class
from dicom import DicomStructureFile

# Import related classes
from types_and_classes import SliceIndexType
from contours import ContourPoints
from contour_plotting import plot_roi_slice

from structure_set import StructureSet
from relations import RelationshipType

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
%matplotlib inline

In [3]:
# Define the path to the test DICOM file
tests_dir = Path.cwd() / r'Tests'
tests_dir = tests_dir.resolve()
test_file_name = 'RS.GJS_Struct_Tests.BRBL BH.dcm'

print(f"=== Loading DICOM file ===: {test_file_name}")
dicom_file = DicomStructureFile(
    top_dir=tests_dir,
    file_name=test_file_name
)
filtered_contours = dicom_file.filter_exclusions()

structure_set = StructureSet(dicom_structure_file=dicom_file)

INFO:dicom:Successfully loaded DICOM dataset from RS.GJS_Struct_Tests.BRBL BH.dcm


=== Loading DICOM file ===: RS.GJS_Struct_Tests.BRBL BH.dcm


INFO:dicom:Extracted 626 contours from 10 ROIs
INFO:dicom:Found 0 frame-of-reference matches and 0 other matches for structure set RS.GJS_Struct_Tests.BRBL BH.dcm
INFO:dicom:Calculated resolution from structure 'BODY': 0.1 cm/pixel
INFO:dicom:Calculated resolution from structure 'BODY': 0.1 cm/pixel
INFO:dicom:Filtered 0 contours from 1 excluded ROIs. Remaining: 626 contours from 9 ROIs
INFO:structure_set:Building StructureSet from 626 contour points
INFO:structure_set:Calculating relationships for 9 structures
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: invalid value encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.

In [13]:
s = structure_set.summary()

In [15]:
s.columns

Index(['ROI', 'Name', 'Physical_Volume', 'Exterior_Volume', 'Hull_Volume',
       'Num_Contours', 'Num_Regions', 'Num_Slices', 'Slice_Range',
       'StructureName', 'DICOM_Type', 'Code', 'CodeScheme', 'CodeMeaning'],
      dtype='object')

In [12]:
row_rois = col_rois = [11, 12, 13, 14, 17, 18, 19, 20]
matrix = structure_set.get_relationship_matrix(row_rois, col_rois,
                                               use_symbols=False)
print(matrix)

Structure_B    Cavity CTV Cavity  eval PTV PTV Cavity     Heart     Lung B  \
Structure_A                                                                  
Cavity         Equals   Overlaps  Overlaps   Overlaps  Disjoint   Disjoint   
CTV Cavity   Overlaps     Equals  Overlaps   Overlaps  Disjoint   Disjoint   
eval PTV     Overlaps   Overlaps    Equals   Overlaps  Disjoint   Disjoint   
PTV Cavity   Overlaps   Overlaps  Overlaps     Equals  Disjoint   Disjoint   
Heart        Disjoint   Disjoint  Disjoint   Disjoint    Equals    Borders   
Lung B       Disjoint   Disjoint  Disjoint   Disjoint   Borders     Equals   
Lung L       Disjoint   Disjoint  Disjoint   Disjoint   Borders   Overlaps   
Lung R       Disjoint   Disjoint  Disjoint   Disjoint   Borders  Partition   

Structure_B    Lung L    Lung R  
Structure_A                      
Cavity       Disjoint  Disjoint  
CTV Cavity   Disjoint  Disjoint  
eval PTV     Disjoint  Disjoint  
PTV Cavity   Disjoint  Disjoint  
Heart         B

In [18]:
# Test the to_dict method
matrix_dict = structure_set.to_dict(row_rois=None, col_rois=None, use_symbols=True)
print("DICOM Types:", matrix_dict['dicom_types'])
print("\nCode Meanings:", matrix_dict['code_meanings'])
print("\nVolumes:", matrix_dict['volumes'])
print("\nNum Regions:", matrix_dict['num_regions'])
print("\nSlice Ranges:", matrix_dict['slice_ranges'])

DICOM Types: {np.int64(9): 'EXTERNAL', np.int64(11): 'GTV', np.int64(12): 'CTV', np.int64(13): '', np.int64(14): '', np.int64(17): '', np.int64(18): '', np.int64(19): '', np.int64(20): ''}

Code Meanings: {np.int64(9): 'Body', np.int64(11): 'Primary Gross Tumor Volume', np.int64(12): 'Clinical Target Volume High Risk', np.int64(13): 'Primary Planning Target Volume', np.int64(14): 'Planning Target Volume High Risk', np.int64(17): 'Heart', np.int64(18): 'Pair of lungs', np.int64(19): 'Left lung', np.int64(20): 'Right lung'}

Volumes: {np.int64(9): 43449.75, np.int64(11): 75.0, np.int64(12): 276.83, np.int64(13): 432.8, np.int64(14): 454.35, np.int64(17): 805.99, np.int64(18): 5189.28, np.int64(19): 2460.92, np.int64(20): 2818.39}

Num Regions: {np.int64(9): 127, np.int64(11): 19, np.int64(12): 29, np.int64(13): 36, np.int64(14): 36, np.int64(17): 41, np.int64(18): 103, np.int64(19): 103, np.int64(20): 99}

Slice Ranges: {np.int64(9): '-13.88 to 14.12', np.int64(11): '-0.62 to 3.62', np.i

In [19]:
# Reload the module to get the latest changes
import importlib
import structure_set
importlib.reload(structure_set)

# Recreate the structure_set object
structure_set_obj = structure_set.StructureSet(dicom_structure_file=dicom_file)

INFO:structure_set:Building StructureSet from 626 contour points
INFO:structure_set:Calculating relationships for 9 structures
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: invalid value encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)


In [20]:
# Test the to_dict method with reloaded module
matrix_dict = structure_set_obj.to_dict(row_rois=None, col_rois=None, use_symbols=True)
print("DICOM Types:", matrix_dict['dicom_types'])
print("\nCode Meanings:", matrix_dict['code_meanings'])
print("\nVolumes:", matrix_dict['volumes'])
print("\nNum Regions:", matrix_dict['num_regions'])
print("\nSlice Ranges:", matrix_dict['slice_ranges'])

DICOM Types: {9: 'EXTERNAL', 11: 'GTV', 12: 'CTV', 13: '', 14: '', 17: '', 18: '', 19: '', 20: ''}

Code Meanings: {9: 'Body', 11: 'Primary Gross Tumor Volume', 12: 'Clinical Target Volume High Risk', 13: 'Primary Planning Target Volume', 14: 'Planning Target Volume High Risk', 17: 'Heart', 18: 'Pair of lungs', 19: 'Left lung', 20: 'Right lung'}

Volumes: {9: np.float64(43449.75), 11: np.float64(75.0), 12: np.float64(276.82), 13: np.float64(432.8), 14: np.float64(454.35), 17: np.float64(805.99), 18: np.float64(5189.28), 19: np.float64(2460.92), 20: np.float64(2818.39)}

Num Regions: {9: 1, 11: 1, 12: 1, 13: 1, 14: 1, 17: 1, 18: 2, 19: 1, 20: 1}

Slice Ranges: {9: '-13.75 to 14.00', 11: '-0.50 to 3.50', 12: '-1.50 to 4.50', 13: '-2.00 to 5.00', 14: '-2.00 to 5.00', 17: '-10.50 to -2.00', 18: '-13.75 to 8.25', 19: '-13.75 to 8.25', 20: '-13.75 to 7.50'}


In [21]:
# Test with another reload after converting numpy floats
import importlib
importlib.reload(structure_set)
structure_set_obj2 = structure_set.StructureSet(dicom_structure_file=dicom_file)
structure_set_obj2.finalize()
structure_set_obj2.calculate_relationships()

result2 = structure_set_obj2.to_dict()
print("Volumes:", result2['volumes'])
print("Num Regions:", result2['num_regions'])

INFO:structure_set:Building StructureSet from 626 contour points
INFO:structure_set:Calculating relationships for 9 structures
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: invalid value encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)
INFO:structure_set:Calculating relationships for 9 structures
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  r

Volumes: {9: 43449.75, 11: 75.0, 12: 276.82, 13: 432.8, 14: 454.35, 17: 805.99, 18: 5189.28, 19: 2460.92, 20: 2818.39}
Num Regions: {9: 1, 11: 1, 12: 1, 13: 1, 14: 1, 17: 1, 18: 2, 19: 1, 20: 1}


In [ ]:
# Test JSON serialization directly
import json

# Get the dict from structure_set_obj2
result_dict = structure_set_obj2.to_dict()

# Try to JSON serialize it
try:
    json_str = json.dumps(result_dict)
    print("✅ JSON serialization successful!")

    # Parse it back to verify
    parsed = json.loads(json_str)
    print("\nDICOM Types from parsed JSON:")
    print(parsed['dicom_types'])
    print("\nVolumes from parsed JSON:")
    print(parsed['volumes'])
except Exception as e:
    print(f"❌ JSON serialization failed: {e}")
    print(f"Error type: {type(e)}")

❌ JSON serialization failed: Object of type int64 is not JSON serializable
Error type: <class 'TypeError'>


In [23]:
# Check what types are in the result dict
result_dict = structure_set_obj2.to_dict()

print("Type of rows:", type(result_dict['rows']))
print("Rows:", result_dict['rows'])
print("Type of first row:", type(result_dict['rows'][0]) if result_dict['rows'] else None)

print("\nType of columns:", type(result_dict['columns']))
print("Columns:", result_dict['columns'])
print("Type of first column:", type(result_dict['columns'][0]) if result_dict['columns'] else None)

print("\nType of matrix data:", type(result_dict['data']))
print("Type of first data row:", type(result_dict['data'][0]) if result_dict['data'] else None)
print("Type of first data element:", type(result_dict['data'][0][0]) if result_dict['data'] and result_dict['data'][0] else None)

Type of rows: <class 'list'>
Rows: [np.int64(9), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]
Type of first row: <class 'numpy.int64'>

Type of columns: <class 'list'>
Columns: [np.int64(9), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]
Type of first column: <class 'numpy.int64'>

Type of matrix data: <class 'list'>
Type of first data row: <class 'list'>
Type of first data element: <class 'str'>


In [24]:
# Reload module again after fixing rows/columns
import importlib
importlib.reload(structure_set)
structure_set_obj3 = structure_set.StructureSet(dicom_structure_file=dicom_file)
structure_set_obj3.finalize()
structure_set_obj3.calculate_relationships()

result3 = structure_set_obj3.to_dict()
print("Rows:", result3['rows'])
print("Type of first row:", type(result3['rows'][0]) if result3['rows'] else None)
print("\nColumns:", result3['columns'])
print("Type of first col:", type(result3['columns'][0]) if result3['columns'] else None)

INFO:structure_set:Building StructureSet from 626 contour points
INFO:structure_set:Calculating relationships for 9 structures
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: invalid value encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)
INFO:structure_set:Calculating relationships for 9 structures
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  r

Rows: [9, 11, 12, 13, 14, 17, 18, 19, 20]
Type of first row: <class 'int'>

Columns: [9, 11, 12, 13, 14, 17, 18, 19, 20]
Type of first col: <class 'int'>


In [25]:
# Final test: JSON serialization
import json

try:
    json_str = json.dumps(result3)
    print("✅ JSON serialization SUCCESSFUL!")

    # Parse back and verify
    parsed = json.loads(json_str)
    print(f"\n✅ Rows: {parsed['rows']}")
    print(f"✅ Columns: {parsed['columns']}")
    print(f"✅ DICOM Types: {parsed['dicom_types']}")
    print(f"✅ Volumes (first 3): {dict(list(parsed['volumes'].items())[:3])}")
    print(f"✅ Colors (first entry): {dict(list(parsed['colors'].items())[:1])}")
except Exception as e:
    print(f"❌ JSON serialization FAILED: {e}")
    import traceback
    traceback.print_exc()

✅ JSON serialization SUCCESSFUL!

✅ Rows: [9, 11, 12, 13, 14, 17, 18, 19, 20]
✅ Columns: [9, 11, 12, 13, 14, 17, 18, 19, 20]
✅ DICOM Types: {'9': 'EXTERNAL', '11': 'GTV', '12': 'CTV', '13': '', '14': '', '17': '', '18': '', '19': '', '20': ''}
✅ Volumes (first 3): {'9': 43449.75, '11': 75.0, '12': 276.82}
✅ Colors (first entry): {'14': [0, 255, 255]}


In [26]:
# Simulate what the FastAPI endpoint would return
# This mimics the MatrixResponse model from main.py

from pydantic import BaseModel
from typing import List, Dict

class MatrixResponse(BaseModel):
    rows: List[int]
    columns: List[int]
    data: List[List[str]]
    row_names: List[str]
    col_names: List[str]
    colors: Dict[int, List[int]]
    dicom_types: Dict[int, str]
    code_meanings: Dict[int, str]
    volumes: Dict[int, float]
    num_regions: Dict[int, int]
    slice_ranges: Dict[int, str]

# Try to create the model with our data
try:
    response = MatrixResponse(**result3)
    print("✅ Pydantic model validation SUCCESSFUL!")

    # Convert to JSON (this is what FastAPI does)
    json_response = response.model_dump_json()
    print("\n✅ FastAPI JSON response generated successfully!")

    # Parse it back (this is what JavaScript receives)
    import json
    received_data = json.loads(json_response)
    print(f"\n✅ JavaScript would receive:")
    print(f"  - ROI 9 Type: {received_data['dicom_types']['9']}")
    print(f"  - ROI 9 Volume: {received_data['volumes']['9']}")
    print(f"  - ROI 18 Regions: {received_data['num_regions']['18']}")

except Exception as e:
    print(f"❌ Pydantic validation FAILED: {e}")
    import traceback
    traceback.print_exc()

✅ Pydantic model validation SUCCESSFUL!

✅ FastAPI JSON response generated successfully!

✅ JavaScript would receive:
  - ROI 9 Type: EXTERNAL
  - ROI 9 Volume: 43449.75
  - ROI 18 Regions: 2


In [27]:
# Let's see EXACTLY what JSON Pydantic generates
import json

response = MatrixResponse(**result3)
json_str = response.model_dump_json()

# Parse it to see the exact structure
data = json.loads(json_str)

print("=== Checking key types in parsed JSON ===")
print(f"Type of 'rows': {type(data['rows'])}")
print(f"Type of first row: {type(data['rows'][0])}")
print(f"First row value: {data['rows'][0]}")
print()
print(f"Type of 'dicom_types': {type(data['dicom_types'])}")
print(f"Keys in dicom_types: {list(data['dicom_types'].keys())[:3]}")
print(f"Type of first key: {type(list(data['dicom_types'].keys())[0])}")
print()
print("=== Testing JavaScript-style access ===")
roi = 9  # This is what JavaScript gets from data.rows[0]
print(f"roi = {roi} (type: {type(roi)})")
print(f"data['dicom_types'][roi] = {data['dicom_types'].get(roi, 'KEY NOT FOUND')}")
print(f"data['dicom_types'][String(roi)] = {data['dicom_types'].get(str(roi), 'KEY NOT FOUND')}")
print()
print("=== Simulating JavaScript logic ===")
dicomType = data['dicom_types'].get(roi) or data['dicom_types'].get(str(roi)) or ''
print(f"Result: '{dicomType}'")

=== Checking key types in parsed JSON ===
Type of 'rows': <class 'list'>
Type of first row: <class 'int'>
First row value: 9

Type of 'dicom_types': <class 'dict'>
Keys in dicom_types: ['9', '11', '12']
Type of first key: <class 'str'>

=== Testing JavaScript-style access ===
roi = 9 (type: <class 'int'>)
data['dicom_types'][roi] = KEY NOT FOUND
data['dicom_types'][String(roi)] = EXTERNAL

=== Simulating JavaScript logic ===
Result: 'EXTERNAL'
